# Aula 06 - Processos e Fontes de Dados

Este notebook é o execício da aula 6 do curso de Ciências de Dados do IFRN para desenvolver um script com a finalidade de carregar dados de fontes distintas (json, API), gravar em um banco de dados SQLite, escrever uma consulta SQL, exportar para outros formatos, demonstrar os processos de verificação érica e legal que pode ser seguido para fazer scrpaing e gerar um gráfico a partir de dados de geo localização.

Exemplo prático: estabelecimentos de saúde do Rio Grande do Norte (CNES)

## 1. Coleta e preparação dos dados

Baixar dados do ministério da saúde, carregar em arquivo JSON, registrar a proveniência (fonte, endpoint, data da coleta, licença), converter para dataframe, fazer limpeza.

In [ ]:
import requests
import datetime
import pandas as pd

# Configurações básicas
endpoint_municipio = "https://apidadosabertos.saude.gov.br/macrorregiao-e-regiao-de-saude/municipio"
endpoint_tipos = "https://apidadosabertos.saude.gov.br/cnes/tipounidades"
endpoint_cnes = "https://apidadosabertos.saude.gov.br/cnes/estabelecimentos"

sigla_uf = "RN"
limit = 10
timeout = 10

print("1. Consultando API de tipos de unidades...")
resp_tipos = requests.get(endpoint_tipos, timeout=timeout)
resp_tipos.raise_for_status()
tipos_unidade_lista = resp_tipos.json().get("tipos_unidade", [])
df_tipos = pd.DataFrame(tipos_unidade_lista)

print("2. Consultando API de municípios...")
resp_mun = requests.get(endpoint_municipio, params={"sigla_uf": sigla_uf, "limit": limit}, timeout=timeout)
resp_mun.raise_for_status()
municipios_lista = resp_mun.json().get("macrorregiao_regiao_saude_municipios", [])

# Preparando a lista de estabelecimentos integrada
lista_estabelecimentos_completa = []

print("3. Consultando API de estabelecimentos para cada município...")
for mun in municipios_lista:
    cod_mun = mun.get("codigo_municipio")
    nome_mun_original = mun.get("municipio", "")
    nome_mun_limpo = nome_mun_original.replace("RN - ", "").strip()

    if not cod_mun:
        continue

    try:
        resp_est = requests.get(endpoint_cnes, params={"codigo_municipio": cod_mun, "limit": limit}, timeout=timeout)
        resp_est.raise_for_status()
        dados_est = resp_est.json().get("estabelecimentos", [])

        for est in dados_est:
            # Vincula o município limpo
            est["municipio_limpo"] = nome_mun_limpo
            lista_estabelecimentos_completa.append(est)
    except Exception as e:
        print(f"Erro ao obter estabelecimentos para o município {nome_mun_limpo}: {e}")

# Criando DataFrame dos estabelecimentos
df_bruto_est = pd.DataFrame(lista_estabelecimentos_completa)

# Mesclar com os tipos de unidade para obter a 'descricao_tipo_unidade'
if not df_bruto_est.empty and not df_tipos.empty:
    df_final = df_bruto_est.merge(df_tipos, on="codigo_tipo_unidade", how="left")
else:
    df_final = df_bruto_est
    df_final["descricao_tipo_unidade"] = "NÃO INFORMADO"

# Remover colunas duplicadas que possam causar conflito no SQLite
df_final = df_final.loc[:, ~df_final.columns.duplicated()]

1. Consultando API de tipos de unidades...
2. Consultando API de municípios...
3. Consultando API de estabelecimentos para cada município...


## 2. Armazenamento e Consulta

Gravar dados em uma tabela SQLite e escreva uma consulta SQL própria usando GROUP BY

In [ ]:
import sqlite3

# 4. Criar o banco de dados SQLite local e salvar as informações
print("4. Gravando no Banco de Dados SQLite...")
conn = sqlite3.connect("cnes_rn_completo.db")
df_final.to_sql("estabelecimentos_completo", conn, index=False, if_exists="replace")

# 5. Executar a consulta SQL pedida pelo usuário
print("5. Executando consulta SQL...")
query = """
SELECT
    municipio_limpo AS municipio,
    COALESCE(descricao_tipo_unidade, 'NÃO INFORMADO') AS descricao_tipo_unidade,
    COUNT(*) AS quantidade_estabelecimentos
FROM
    estabelecimentos_completo
WHERE
    descricao_tipo_unidade IN ('HOSPITAL GERAL', 'CENTRO DE SAUDE/UNIDADE BASICA')
GROUP BY
    municipio_limpo,
    descricao_tipo_unidade
ORDER BY
    municipio ASC,
    quantidade_estabelecimentos DESC;
"""

df_resultado_cnes = pd.read_sql_query(query, conn)
conn.close()

print("\n=== RESULTADO DA CONSULTA SQL (AMOSTRA DOS DADOS) ===")
display(df_resultado_cnes)

4. Gravando no Banco de Dados SQLite...
5. Executando consulta SQL...

=== RESULTADO DA CONSULTA SQL (AMOSTRA DOS DADOS) ===


,municipio,descricao_tipo_unidade,quantidade_estabelecimentos
0,ACARI,CENTRO DE SAUDE/UNIDADE BASICA,6
1,ACARI,HOSPITAL GERAL,1
2,AFONSO BEZERRA,HOSPITAL GERAL,2
3,AFONSO BEZERRA,CENTRO DE SAUDE/UNIDADE BASICA,2
4,AGUA NOVA,CENTRO DE SAUDE/UNIDADE BASICA,2
5,ALEXANDRIA,CENTRO DE SAUDE/UNIDADE BASICA,2
6,ALEXANDRIA,HOSPITAL GERAL,1
7,ALMINO AFONSO,CENTRO DE SAUDE/UNIDADE BASICA,2
8,ALMINO AFONSO,HOSPITAL GERAL,1
9,ALTO DO RODRIGUES,CENTRO DE SAUDE/UNIDADE BASICA,4


## 3. Exportar o conjunto de dados em CSV e parquet, e compare o tamanho dos arquivos gerados.

In [ ]:
import os
import pandas as pd

# Definir os nomes dos arquivos
arquivo_csv = "estabelecimentos_completo.csv"
arquivo_parquet = "estabelecimentos_completo.parquet"

# 1. Exportar os dados para CSV e Parquet
print("1. Exportando dados do DataFrame...")
df_final.to_csv(arquivo_csv, index=False)
df_final.to_parquet(arquivo_parquet, index=False)

# 2. Obter o tamanho dos arquivos em disco
tamanho_csv = os.path.getsize(arquivo_csv)
tamanho_parquet = os.path.getsize(arquivo_parquet)

# 3. Exibir os resultados e fazer o comparativo
print("\n=== COMPARAÇÃO DE TAMANHO DE ARQUIVOS ===")
print(f"Arquivo CSV: {tamanho_csv} bytes ({tamanho_csv / 1024:.2f} KB)")
print(f"Arquivo Parquet: {tamanho_parquet} bytes ({tamanho_parquet / 1024:.2f} KB)")

diferenca_porcentagem = ((tamanho_csv - tamanho_parquet) / tamanho_csv) * 100
print(f"\nO formato Parquet é maior que o formato CSV para esta base de dados porque são poucos dados, mas se forem uma base maior deve ficar menor.")

1. Exportando dados do DataFrame...

=== COMPARAÇÃO DE TAMANHO DE ARQUIVOS ===
Arquivo CSV: 35056 bytes (34.23 KB)
Arquivo Parquet: 40376 bytes (39.43 KB)

O formato Parquet é maior que o formato CSV para esta base de dados porque são poucos dados, mas se forem uma base maior deve ficar menor.


# Checklist de Verificação Ética e Legal para Web Scraping

Etapas essenciais que devem ser seguidas antes de iniciar a coleta (por exemplo, **Wikipedia**).

- [ ] Localizar o arquivo `robots.txt`.
- [ ] Localizar o site links como "Termos de Uso", "Condições de Serviço" ou "Políticas de Privacidade".
- [ ] Verifique se há algo explicando sobre o uso de robôs, scrapers, spiders ou qualquer método automatizado de extração de dados.
- [ ] Descubra se as informações do site são protegidas por direitos autorais proprietários ou sob licenças abertas (como Creative Commons, licenças governamentais, etc.).
- [ ] Confirme se a finalidade da sua análise de dados está de acordo com as permissões da licença do conteúdo.
- [ ] Caso a licença exija atribuição, registre em seus metadados a fonte original e os direitos aplicados.
- [ ] Evite sobrecarregar o servidor de destino. Insira atrasos (`time.sleep()`) entre as requisições para simular um comportamento humano.
- [ ] Evite extrair dados que possam identificar indivíduos diretamente (como nomes, telefones, e-mails pessoais ou CPFs) sem o consentimento ou base legal explícita.
- [ ] Sempre priorize o uso de endpoints de API oficiais fornecidos pelo desenvolvedor antes de recorrer ao scraping do HTML bruto.